# 5.9 - Causal ML: does a characteristic move returns, or travel with what does?

> **Needs:** notebook 5.2's features.
>
> **Produces:** its estimates, kept in the artifact store.

Everything so far has asked whether a signal *predicts* returns. A different question matters for understanding, and
for trusting a signal to keep working: does the characteristic itself carry the effect, or does it only travel with
other characteristics that do? Momentum, for example, is correlated with size, volatility, profitability and
liquidity; its apparent effect might be theirs.

*Machine Learning for Trading* (chapter 15) answers with **double machine learning** (DML; Chernozhukov et al., 2018).
This notebook estimates:

1. the effect of 12-1 month momentum on next month's return, controlling flexibly for the other characteristics of
   notebook 5.2 - against the naive estimate (no controls) and a linear one (straight-line controls);
2. a **factor-zoo check** (after Feng, Giglio and Xiu, 2020, who asked which of hundreds of published factors add
   anything once the others are accounted for): each of the features that survived or nearly survived 5.2's triage,
   as the treatment, with all the others as controls, under false-discovery control.

## Double machine learning, step by step

The **partially linear model**: next month's excess return $y$ depends on the treatment $d$ (here a centred
percentile, so its effect $\theta$ is the return gap between the lowest and the highest company) linearly, and on the
other characteristics $X$ in any way at all:
$$y = \theta\, d + g(X) + \varepsilon, \qquad d = m(X) + v.$$

1. **Predict both from the controls.** Fit flexible models (gradient-boosted trees) of $\ell(X) = E[y \mid X]$ and
   $m(X) = E[d \mid X]$.
2. **Cross-fit.** Cut the months into five consecutive blocks (purged of overlapping labels); each block's
   predictions come from models fitted on the other four. A model never predicts the rows it learned from - otherwise
   its overfitting would leak into $\theta$.
3. **Regress residual on residual.** $\hat\theta = \sum \hat v \hat e / \sum \hat v^2$, with $\hat e = y - \hat\ell(X)$
   and $\hat v = d - \hat m(X)$: the part of the return the controls cannot explain, on the part of the treatment
   they cannot explain. Errors in the two predictions enter only multiplied together, so a moderately wrong model
   barely biases $\theta$ (*Neyman orthogonality*).
4. **Standard errors clustered by month.** Companies in the same month share shocks; summing the estimating
   equation within each month before taking its variance allows for that.

**What "causal" means here:** the estimate is the treatment's effect *if every confounder is among the controls*. A
missing one (an unmeasured risk, investor flows) would bias it like any regression. DML's contribution is to control
for the measured ones flexibly without adding the bias that flexible fitting usually brings.

In [1]:
import pandas as pd
from IPython.display import display

from equity_research.data import artifacts as store
from equity_research.data.cache import ResearchCache
from equity_research.data.universe import load_universe_data
from equity_research.ml import causal as cz
from equity_research.ml import linear as ln
from research_core import charts
from research_core.analytics.signals import labels as lb
from research_core.analytics.signals import validation as va

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
charts.style()

UNIVERSE, START, END = "SP1500", "2005-01-01", "2024-12-31"  # END: the last day before the hold-out
FDR = 0.10
cache = ResearchCache()
SNAPSHOT = store.data_snapshot(cache)
data = load_universe_data(cache, UNIVERSE, START, END)
dates = data.rebalances
FEATURES = {"universe": UNIVERSE, "start": START, "end": END, "dates": "month ends", "version": 1}
long = store.load("features", "company_month_end", FEATURES, snapshot=SNAPSHOT)
assert long is not None, "run notebook 5.2 first: it builds the features for this data snapshot"
monthly = store.cached(
    "labels",
    "fwd_ret_month",
    {"universe": UNIVERSE, "start": START, "end": END, "dates": "rebalances"},
    lambda: lb.period_returns(data.returns, dates),
    snapshot=SNAPSHOT,
)
x = ln.centred_ranks(long)
y = lb.trimmed_excess(monthly, data.members)
labelled = dates[dates < dates[-1]]  # the last month end has no next month
ends = va.next_date_ends(dates)
blocks = va.purged_kfold(labelled, ends, k=5, embargo=1)
# momentum's own family: measures of the same past price path, which would absorb the effect being measured
TREND = ["mom_12_1", "mom_6_1", "resid_mom", "trend_slope", "trend_gap", "ffd_price", "break_stat"]
controls = [c for c in x.columns if c not in TREND]
print(f"{len(controls)} controls; {len(blocks)} cross-fitting blocks of about {len(blocks[0].test)} months")

left out 1078 facts of 2 companies in ['GBP']


38 controls; 5 cross-fitting blocks of about 48 months


## Step 1 - the effect of momentum

Three estimates of $\theta$ for 12-1 momentum, over all months 2005-2024 and in each half: with no controls, with the
controls entering as straight lines (ordinary least squares), and by DML with boosted trees. The controls exclude
momentum's own family (six-month and residual momentum, the trend and break features): they measure the same past
price path, and controlling for them would remove the very effect being measured.

In [2]:
def estimates(window: pd.DatetimeIndex) -> pd.DataFrame:
    """Momentum's effect three ways on the months of ``window``."""
    part = x[x.index.get_level_values("date").isin(window)]
    sub_blocks = va.purged_kfold(window, ends, k=5, embargo=1)
    return pd.DataFrame(
        {
            "no controls": cz.ols_effect(part, y, "mom_12_1"),
            "linear controls": cz.ols_effect(part, y, "mom_12_1", controls),
            "double ML": cz.dml(part, y, treatment="mom_12_1", controls=controls, blocks=sub_blocks),
        }
    ).T


halves = {
    "2005-2024": labelled,
    "2005-2014": labelled[labelled.year <= 2014],
    "2015-2024": labelled[labelled.year >= 2015],
}
momentum = store.cached(
    "causal",
    "momentum",
    FEATURES | {"controls": controls, "k": 5},
    lambda: pd.concat({k: estimates(v) for k, v in halves.items()}),
    snapshot=SNAPSHOT,
)
display(
    momentum.style.format({"theta": "{:+.4f}", "se": "{:.4f}", "t": "{:+.2f}", "p_value": "{:.3f}", "rows": "{:,.0f}"})
)

## Step 2 - the factor zoo: which features add something of their own?

The two discoveries and the seven near-misses of 5.2's triage, each in turn as the treatment, with every other feature
as a control (momentum's family included this time: the question is whether each adds to *everything* else). DML with
boosted trees, over 2005-2024; Benjamini-Hochberg at 10 % across the nine.

In [3]:
ledger = store.load(
    "triage", "company_features_month", FEATURES | {"label": "fwd_ret_month", "fdr": FDR}, snapshot=SNAPSHOT
)
assert ledger is not None, "run notebook 5.2 first: it keeps the triage ledger"
candidates = list(ledger.index[ledger["discovery"] | (ledger["t_nw"].abs() >= 1.99)])
zoo = store.cached(
    "causal",
    "factor_zoo",
    FEATURES | {"candidates": candidates, "k": 5, "fdr": FDR},
    lambda: cz.factor_zoo(x, y, candidates, blocks, q=FDR),
    snapshot=SNAPSHOT,
)
zoo["alone (6.2)"] = ledger.loc[zoo.index, "mean_ic"]
zoo["alone t (6.2)"] = ledger.loc[zoo.index, "t_nw"]
display(
    zoo.sort_values("t", key=abs, ascending=False).style.format(
        {
            "theta": "{:+.4f}",
            "se": "{:.4f}",
            "t": "{:+.2f}",
            "p_value": "{:.4f}",
            "rows": "{:,.0f}",
            "alone (6.2)": "{:+.4f}",
            "alone t (6.2)": "{:+.2f}",
        }
    )
)

,theta,se,t,p_value,rows,discovery,alone (6.2),alone t (6.2)
characteristic,,,,,,,,
zero_days_3m,-0.0024,0.0008,-2.97,0.0029,"358,706",True,-0.0077,-2.23
idio_vol_3m,-0.0065,0.0027,-2.38,0.0175,"358,706",True,-0.0248,-2.49
RoE,-0.0065,0.0040,-1.64,0.1006,"358,706",False,+0.0147,+2.35
sales_growth,+0.0019,0.0015,+1.22,0.2216,"358,706",False,+0.0126,+2.19
share_change,-0.0010,0.0012,-0.84,0.4001,"358,706",False,-0.0154,-3.53
har_vol,-0.0045,0.0078,-0.58,0.5624,"358,706",False,-0.0246,-1.99
max_ret_1m,-0.0012,0.0023,-0.54,0.5917,"358,706",False,-0.0203,-2.04
Op. In./(NWC+FA),-0.0003,0.0018,-0.17,0.8614,"358,706",False,+0.0171,+2.45
ROCE,+0.0007,0.0057,+0.12,0.9028,"358,706",False,+0.0197,+2.94


## Findings

*Rerun on 17 November 2026 on quant_research_local (history from 2000). 5.2's triage now passes nine
characteristics here, not eleven - EBIT over assets and the GARCH forecast fell just short of p 0.05 - and two of the
nine keep an effect of their own where three of eleven did (the GARCH forecast was one of the three).*

**1. Momentum has no effect on next month's return here, however it is measured (step 1).** Over 2005-2024 the gap
in next month's excess return between the lowest- and the highest-momentum company:

* **No controls** - -0.09 % (t -0.21).
* **Straight-line controls** - -0.08 % (t -0.22).
* **Double ML** - -0.08 % (t -0.23).

Every half-period estimate has |t| below 0.6. The standard error, about 0.34 % a month, rules out any effect larger
than roughly ±0.7 % a month with 95 % confidence. This agrees with notebook 5.2, where momentum's IC was +0.006
(t +0.57). In the S&P 1500 over these twenty years, a month's momentum ranking did not move the next month's return.
That holds alone, beside the other characteristics, and whether they enter as straight lines or flexibly. The
controls change the estimate by a few hundredths of a percent, so momentum's absence is not hidden by confounding
in either direction. (Its long-run premium in older and broader samples is another question; this is the S&P 1500
since 2005, a month ahead.)

**2. The factor zoo changes the picture of 5.2 (step 2).** With every other feature controlled for flexibly, two
of the nine characteristics keep an effect of their own at a 10 % false-discovery rate - both about volatility or
illiquidity:

| Characteristic | Effect, lowest to highest (a month) | t | Alone in 5.2 (IC, t) |
|---|---|---|---|
| zero-return days (3 months) | -0.24 % | -2.97 | -0.008, -2.23 |
| idiosyncratic volatility (3 months) | -0.65 % | -2.38 | -0.025, -2.49 |

The two discoveries of 5.2's triage lose significance:

* **Share-count change** - -0.10 % (t -0.84): pointing the same way as before, but no longer distinguishable from
  zero once the other characteristics are held fixed.
* **ROCE** - +0.07 % (t +0.12).

Return on equity changes sign: +0.015 alone, -0.65 % (t -1.64) with the others controlled for.

**3. Reading this correctly.** "Keeps an effect of its own" means an effect beyond everything else measured, and
"loses it" does not mean the characteristic is useless.

* **Profitability** - ROCE sits beside return on equity, EBIT over assets and return on operating capital, which
  rank companies much alike. Holding the others fixed while ROCE moves leaves little variation of its own, so its
  separate effect is unidentifiable rather than absent. The family's shared effect is what 5.2's triage found.
* **Volatility** - the same caution applies, more sharply. Moving idiosyncratic volatility from lowest to highest
  while holding plain volatility and the GARCH and HAR forecasts fixed is an extrapolation: those companies barely
  exist. Which volatility measure carries the effect depends on which enter the test - in the run of 14 November,
  with the GARCH forecast among the nine near-misses, it took the largest share. What the result does say is robust:
  some measure of volatility, and of how rarely a share trades, predicts lower returns over and above everything else
  in the set. Profitability and issuance do not, once volatility and liquidity are accounted for.

**4. What it means for the plan.** The repository's two "surviving" factors from 5.2, issuance and quality
(through ROCE), add little once volatility and liquidity are in the model. A linear model of the full feature set
already had that information (5.3's ridge leaned on the GARCH forecast and the liquidity measures). The consistent
thread across 5.2, 5.3 and this notebook is the low-volatility and illiquidity effect: calm, frequently traded shares
did better. That is a known anomaly, and it also explains why 5.2's percentile models tilted towards calm shares.
None of this is causal proof: an unmeasured confounder, such as the demand of investors who prefer lottery-like
shares, could produce the same pattern.

## Glossary

* **Confounder** - a variable that affects both the treatment and the outcome, so their correlation is not the
  treatment's effect.
* **Partially linear model** - the outcome depends linearly on the treatment and in any way on the controls.
* **Cross-fitting** - predicting each part of the data with models fitted on the other parts.
* **Neyman orthogonality** - a property of an estimating equation that makes it insensitive, to first order, to errors
  in the models it relies on.
* **Clustered standard error** - one that allows observations in the same group (here the same month) to share shocks.
* **Factor zoo** - the hundreds of characteristics published as predicting returns, most of them related to others.